# Phase 3: train and test the baseline model in Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bhavilkawdia5-ux/leaf_disease_detection/blob/main/ml/notebooks/phase3_baseline.ipynb)

This runs Phase 3 of `docs/SYSTEM_DESIGN.md` on this Colab machine. It:

1. downloads and merges the public datasets, as in `phase2_dataset.ipynb`,
2. fine-tunes MobileNetV3 at 224x224 with `ml/src/train.py` (no lighting augmentation),
3. tests it with `ml/src/evaluate.py` on the test split, as it is and with synthetic
   shadow, harsh sun and low light,
4. writes `ml/reports/baseline.md` and lets you download it with the model.

**Use a GPU:** Runtime > Change runtime type > T4 GPU. It does not use Google Drive, so
each new session downloads again (about 30 minutes) and training takes about 30 to 60
minutes more. Run each cell with **Shift + Enter**, top to bottom.

## 1. Get the code

In [ ]:
import os
if os.path.isdir("/content/leaf_disease_detection"):
    !git -C /content/leaf_disease_detection pull -q
else:
    !git clone -q https://github.com/bhavilkawdia5-ux/leaf_disease_detection.git /content/leaf_disease_detection
%cd /content/leaf_disease_detection
!git log --oneline -1

## 2. Install packages and check the GPU

TensorFlow is already in Colab. If no GPU is listed, change the runtime type (see the
top) and run the cells again.

In [ ]:
!pip install -q pyyaml pyarrow pillow
import tensorflow as tf
print("TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none - training will be very slow")

## 3. Download and merge the datasets

About 15 to 30 minutes. If a dataset says FAILED, run the cell again. The counts should
match `docs/datasets.md` (18,455 images on 2026-10-17).

In [ ]:
!python ml/src/download.py --data /content/data
!python ml/src/dataset.py --data /content/data | tee /content/phase3_counts.txt

## 4. Quick check

Trains on 20 images per class for 2 epochs and tests on 20 per class, about 3 minutes.
It only checks that the scripts run end to end; ignore its accuracy.

In [ ]:
!python ml/src/train.py --data /content/data --out /content/models_check --limit 20 --epochs-head 1 --epochs-fine 1
!python ml/src/evaluate.py --data /content/data --models /content/models_check --report /content/check.md --limit 20

## 5. Train the baseline

About 30 to 60 minutes on a T4. Stage 1 trains only the last layer (5 epochs), stage 2
fine-tunes the top of MobileNetV3 (up to 10 epochs, stops early). The best epoch by
validation accuracy is kept. Keep this tab open so Colab does not disconnect.

In [ ]:
!python ml/src/train.py --data /content/data --out /content/models

## 6. Test per lighting condition

About 5 to 10 minutes. Writes `ml/reports/baseline.md`.

In [ ]:
!python ml/src/evaluate.py --data /content/data --models /content/models --report ml/reports/baseline.md
from IPython.display import Markdown, display
display(Markdown(open("ml/reports/baseline.md", encoding="utf-8").read()))

## 7. Download the results

Downloads `baseline.md`, `train_info.json`, `labels.txt` and `baseline.keras` (about
15 MB). Your browser may ask to allow several downloads.

In [ ]:
from google.colab import files
for path in ["ml/reports/baseline.md", "/content/models/train_info.json",
             "/content/models/labels.txt", "/content/models/baseline.keras"]:
    files.download(path)

## 8. What to do next

1. Put `baseline.md` in `ml/reports/` and `baseline.keras`, `labels.txt` and
   `train_info.json` in `ml/models/` (git-ignored) on your laptop.
2. Tell Claude Code it is there. Phase 3 is done when `baseline.md` shows accuracy per
   lighting condition and the drop under shadow and harsh sun is measured.
3. If a cell fails, copy its output and paste it to Claude Code.